# Semantic Filtering of the Arabic-English Corpus

This notebook demonstrates the public filtering pipeline. The reusable implementation is in `scripts/filter_parallel_corpus.py`.

The pipeline:

1. Streams the source Hugging Face dataset.
2. Normalizes whitespace and rejects empty pairs.
3. Removes exact duplicate sentence pairs using a disk-backed hash index.
4. Computes aligned multilingual Sentence-BERT cosine similarity in batches.
5. Writes accepted pairs, rejected pairs, errors, and a reproducibility summary.

The default retained range is **0.70–0.99**, inclusive. Change these values only when documenting a new experimental configuration.

## Environment

Create a virtual environment from the repository root and install `requirements.txt` before running this notebook.

```bash
python -m venv .venv
pip install -r requirements.txt
```

For exact reproduction, set `DATASET_REVISION` to the Hugging Face commit hash used in the original experiment.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "filter_demo"
DATASET_REVISION = None  # Replace with the exact source revision for publication runs.
LIMIT = 1000             # Set to None for the complete split.
OUTPUT_DIR

In [ ]:
command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "filter_parallel_corpus.py"),
    "--output-dir", str(OUTPUT_DIR),
    "--min-score", "0.70",
    "--max-score", "0.99",
    "--overwrite",
]
if DATASET_REVISION:
    command.extend(["--dataset-revision", DATASET_REVISION])
if LIMIT is not None:
    command.extend(["--limit", str(LIMIT)])

print(" ".join(command))
# Uncomment to execute:
# subprocess.run(command, check=True)

## Inspect the run summary

The summary records thresholds, model name, source revision, package versions, timestamps, and processing counts.

In [ ]:
summary_path = OUTPUT_DIR / "filtering_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    summary
else:
    print("Run the command cell first; no summary file exists yet.")

In [ ]:
import csv

accepted_path = OUTPUT_DIR / "accepted_pairs.tsv"
if accepted_path.exists():
    with accepted_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle, delimiter="\t"))
    rows[:5]
else:
    print("No accepted-pairs file exists yet.")